In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch
from langchain_core.tools import tool 
import requests
import math
import os

In [2]:
load_dotenv()

True

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    api_key=GROQ_API_KEY
)
llm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000286542AB110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028656FFA510>, model_name='llama-3.3-70b-versatile', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
## tool

search_tool = TavilySearch(
    tavily_api_key=TAVILY_API_KEY, 
    max_results=5,
    topic="general",
    search_depth="advanced"
)


@tool
def calculator(expression: str)-> str:
    """
    Useful for a simple math calculation. 
    Input should be a valid math expression. 
    Ex.: 2+2, sqrt(16), 10*5 
    """

    try:
        allowed = {
            "math" : math,
            "abs": abs,
            "round" : round,
            "min" : min,
            "max" : max,
            "sum" : sum}
        
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    
    except Exception as e:
        return f"calculation error : {str(e)}"

        
@tool
def get_stock_price(symbol: str)-> dict:
    """
    Fetch the latest stock price for a given symbol. e.g.('AAPL', 'TSLA')
    using Alpha Vantage with API Key in URL.
    """

    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=demo"


    